In [13]:
import sys
sys.path.append('../scripts')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

from pathlib import Path
from train_cnn import SimpleCNN

I0000 00:00:1774643760.717032  178430 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [14]:
model_path = str(Path("../models/cnn_model.pt").resolve())

In [15]:
if not os.path.exists(model_path):
    print(f"❌ Model file not found: {model_path}")
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Loading model from {model_path} on device: {device}")
    loaded_obj = torch.load(model_path, map_location=device)
    
    # Check if it's a state_dict (OrderedDict) or full model
    if isinstance(loaded_obj, dict):
        # It's a state_dict - need to reconstruct the model
        # Load model metadata from training_info.json if available
        info_path = model_path.replace('cnn_model.pt', 'training_info.json').replace('.pt', '_info.json')
        if not os.path.exists(info_path):
            # Try alternate path
            base_dir = os.path.dirname(model_path)
            info_path = os.path.join(base_dir, 'training_info.json')
        
        if os.path.exists(info_path):
            import json
            with open(info_path, 'r') as f:
                info = json.load(f)
                input_shape = info.get('input_shape', [128, 128])
                num_classes = info.get('num_classes', 2)
        else:
            # Default values
            input_shape = [128, 128]
            num_classes = 2
        
        model = SimpleCNN(input_shape, num_classes)
        model.load_state_dict(loaded_obj)
    else:
        # It's already a full model object
        model = loaded_obj
    
    model.to(device)
    model.eval()
    print(f"✅ Model loaded successfully!")

Loading model from /share/users/student/s/ssahu/aigm-classifier/models/cnn_model.pt on device: cpu
✅ Model loaded successfully!


In [16]:
print(model)

SimpleCNN(
  (reshape): Identity()
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)
